# Tendencia de PM2.5 e Segmentacao de Paises

In [ ]:
import sys
sys.path.append('..')
import matplotlib.pyplot as plt

from src.data.load_data import pipeline_completo
from src.features.engenharia_features import calcular_tendencia_pm25
from src.models.segmentacao_paises import preparar_dados_cluster, escolher_k, treinar_kmeans

pm25, snapshot = pipeline_completo()
tendencia = calcular_tendencia_pm25(pm25)
print('paises com tendencia calculada:', len(tendencia))
tendencia.sort_values('tendencia_pm25_ano').head(5)

In [ ]:
completo = snapshot.merge(tendencia[['pais_iso3', 'tendencia_pm25_ano']], on='pais_iso3', how='inner')
X_escalado, _ = preparar_dados_cluster(completo)
tabela_k = escolher_k(X_escalado)
tabela_k

In [ ]:
melhor_k = int(tabela_k.loc[tabela_k['silhouette'].idxmax(), 'k'])
modelo = treinar_kmeans(X_escalado, melhor_k)

fig, ax = plt.subplots()
ax.scatter(completo['pm25'], completo['tendencia_pm25_ano'], c=modelo.labels_, cmap='viridis', s=25, alpha=0.7)
ax.axhline(0, color='gray', linestyle='--', linewidth=1)
ax.set_xlabel('PM2.5 medio, 2021')
ax.set_ylabel('Tendencia anual de PM2.5 (2010-2023)')
ax.set_title('Segmentos de paises por perfil de qualidade do ar')
plt.tight_layout()
plt.savefig('../reports/figures/segmentos_paises.png', dpi=150)
plt.show()

## Conclusoes

A China aparece com a maior melhora de PM2.5 no periodo (tendencia de -2,8 ug/m3 por ano), resultado de politicas ambientais fortes na ultima decada, algo bem documentado. Nigeria e outros paises da Africa Ocidental aparecem entre os que mais pioraram. A segmentacao separa paises que ja tem ar limpo e estavel, dos que tem ar ruim mas melhorando, dos que tem ar ruim e piorando, o que ajuda a priorizar politica publica.